In [ ]:
# GE Title: NOAA CDR OISST v02r01: Optimum Interpolation Sea Surface Temperature

# https://developers.google.com/earth-engine/datasets/catalog/NOAA_CDR_OISST_V2_1

# Description: The NOAA 1/4 degree daily Optimum Interpolation Sea Surface Temperature (OISST) provides complete ocean temperature fields constructed by combining bias-adjusted observations from different platforms (satellite, ships, buoys) on a regular global grid, with gaps filled in by interpolation. Satellite data from the Advanced Very High Resolution Radiometer (AVHRR) provides the main input which permits the high temporal-spatial coverage beginning in late 1981 to the present.

# The OISST dataset has a single day's data processed twice. First a near real-time preliminary version is released with a lag of 1 day, and a final version with a lag of 14 days. The final version uses extra days for smoothing, and zonal bias correction in addition to replacing the preliminary version.

# citation: Richard W. Reynolds, Viva F. Banzon, and NOAA CDR Program (2008): NOAA Optimum Interpolation 1/4 Degree Daily Sea Surface Temperature (OISST) Analysis, Version 2. [indicate subset used]. NOAA National Centers for Environmental Information. doi:10.7289/V5SQ8XB5 [access date].

# DOI: https://doi.org/10.7289/V5SQ8XB5

# Terms of Use: The NOAA CDR Program's official distribution point for CDRs is NOAA's National Climatic Data Center which provides sustained, open access and active data management of the CDR packages and related information in keeping with the United States' open data policies and practices as described in the President's Memorandum on "Open Data Policy" and pursuant to the Executive Order of May 9, 2013, "Making Open and Machine Readable the New Default for Government Information". In line with these policies, the CDR data sets are nonproprietary, publicly available, and no restrictions are placed upon their use. For more information, see the Fair Use of NOAA's CDR Data Sets, Algorithms and Documentation pdf.

# availability: 1981 to  2025

# provider: NOAA

# Cadence: 1 Day

# Pixel size: 27830 meters

# ee snippet: ee.ImageCollection("NOAA/CDR/OISST/V2_1")

# band of use: anom

# band information - 	Units: Celsius, min:-1887*, max:1902* (* = estimated), scale:0.01
# Temperature anomaly; the daily OISST minus a 30-year climatological mean.

# helpful band: sst, daily sea surface temp

# lets visualize this image set with guaymas office, office number 2604

# generally, its suggested to take a point and turn it into an ee object then make the buffer

In [ ]:
# import library
import ee
import geemap
import geopandas as gpd
import pandas as pd
import time


# authenticate & initilialize earth engine
ee.Authenticate()
ee.Initialize(project='ee-sst-j-felix')

In [ ]:
# get the boundary for an office
# load the shapefile
office = gpd.read_file('/content/drive/MyDrive/BIENPESCA/shapefiles/office_points_latest.shp')

office.columns = ["state_id",  "state",    "office_id",  "office",   "locality",  "cvegeo",   "status",   "stat_abr",  "mun_id",
                          "local_id",  "climate",  "latitud", "longitud",  "altitud",  "letter_id",  "population",  "male_pop",  "feml_pop",
                          "opccupied_households",  "obs_id",   "municipio",  "geometry"]


In [ ]:
# get the image collection (filter for date)
anom_collection = (
    ee.ImageCollection("NOAA/CDR/OISST/V2_1")
    .filterDate('2006-01-01', '2024-12-31')
    .select('anom')
)

In [ ]:
# Convert office into ee.FeatureCollection with 100 km buffe
def create_buffered_feature(row):
    coords = row.geometry.coords[0]
    ee_point = ee.Geometry.Point(coords)
    buffer = ee_point.buffer(100000)  # 100 km
    return ee.Feature(buffer).set({'office_id': row.office_id})

# Apply the conversion
features = [create_buffered_feature(row) for idx, row in office.iterrows()]
fc = ee.FeatureCollection(features)

In [ ]:
# Define SST anomaly extraction function
def extract_monthly_stats(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    mean_img = anom_collection.filterDate(start, end).mean()

    def extract_for_feature(f):
        mean = mean_img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=f.geometry(),
            scale=1000,
            maxPixels=1e8
        )
        return f.set({
            'year': year,
            'month': month,
            'avg_sst_anomaly': mean.get('anom')
        })

    return fc.map(extract_for_feature)

In [ ]:
#  Aggregate over all months
years = list(range(2006, 2025))
months = list(range(1, 13))

results = []
for y in years:
    for m in months:
        results.append(extract_monthly_stats(y, m))

# Merge all feature collections into one
all_months_fc = ee.FeatureCollection(results).flatten()

In [ ]:
# Export as CSV to Google Drive
import time

task = ee.batch.Export.table.toDrive(
    collection=all_months_fc,
    description='monthly_sst_anomaly_export',
    driveFolder='BIENPESCA_exports',
    fileNamePrefix='sst_anomaly_all_offices_2006_2024',
    fileFormat='CSV'
)
task.start()


while task.active():
    print("Exporting... status:", task.status()['state'])
    time.sleep(30)

print("Export complete.")

Exporting... status: READY
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... status: RUNNING
Exporting... sta